# LANL 00 Load

Repo-native inspection notebook for the LANL Earthquake Prediction dataset.
This pipeline stays inside the `lanl` namespace and keeps the proxy-state assumption explicit.


In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import LANL_CONFIG
from src.datasets.lanl import build_lanl_summary
from src.io.lanl import load_lanl_train
from src.preprocess.lanl import clean_lanl_dataframe
from src.utils.plotting import plot_signal_panel

RESULTS_DIR = LANL_CONFIG.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INSPECTION_ROWS = 200_000
PLOT_ROWS = 20_000


In [2]:
summary = build_lanl_summary(nrows=INSPECTION_ROWS)
summary_path = RESULTS_DIR / "dataset_summary.json"
inventory_path = RESULTS_DIR / "raw_file_inventory.json"
readme_path = RESULTS_DIR / "README.md"
subset_path = RESULTS_DIR / "inspection_subset.csv"

summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
readme_path.write_text(
    "\n".join(
        [
            "# LANL Dataset Summary",
            "",
            f"- Source URL: {summary['dataset']['source_url']}",
            f"- Raw file: {summary['raw_file']}",
            "- State variables: tau_proxy, V_proxy",
            "- Velocity is derived from the smoothed acoustic proxy",
            f"- Segmentation rule: {summary['dataset']['segmentation']['strategy']}",
        ]
    )
    + "\n",
    encoding="utf-8",
)

inspection_df = clean_lanl_dataframe(load_lanl_train(nrows=INSPECTION_ROWS))
inspection_df.to_csv(subset_path, index=False)
inventory = {
    "dataset": "lanl",
    "raw_dir": str(LANL_CONFIG.raw_dir),
    "filenames_found": sorted(path.name for path in LANL_CONFIG.raw_dir.iterdir() if path.is_file()),
    "file_types_found": sorted({path.suffix.lower() or "<no_extension>" for path in LANL_CONFIG.raw_dir.iterdir() if path.is_file()}),
    "loader": "load_lanl_train",
    "loader_status": "ok",
    "loader_raw_file": summary["raw_file"],
    "parsed_columns": summary["schema"]["columns"],
    "available_variables": summary["schema"]["columns"],
}
inventory_path.write_text(json.dumps(inventory, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print(inspection_df.head())


{
  "dataset": {
    "name": "lanl",
    "label": "LANL Earthquake Prediction",
    "source_url": "https://www.kaggle.com/competitions/LANL-Earthquake-Prediction",
    "raw_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\data\\lanl",
    "results_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\results\\lanl",
    "notebooks_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\notebooks\\lanl",
    "preferred_raw_names": [
      "train.csv",
      "lanl_train.csvtdunczn5.part"
    ],
    "raw_globs": [
      "*.csv",
      "*.part"
    ],
    "column_aliases": {
      "time": [
        "time"
      ],
      "acoustic_data": [
        "acoustic_data"
      ],
      "time_to_failure": [
        "time_to_failure"
      ],
      "state_1": [
        "tau_proxy"
      ],
      "state_2": [
        "V_proxy"
      ]
    },
    "velocity_mode": "proxy_from_smoothed_acoustic_data",
    "smoothing": {
      "window": 101,
      "polyorder": 3
    },
    "segmentation": {
      "stra

In [3]:
plot_df = inspection_df.iloc[:PLOT_ROWS].copy()
plot_signal_panel(plot_df, "time", ["acoustic_data", "time_to_failure"], "LANL inspection subset")
